# Fase 3 — Ingeniería de prompts (EcoMarket)

Este notebook sigue la misma idea práctica que el proyecto de referencia **prompt-engineering-sample** (`app.py` + API de OpenAI): se define un **sistema de mensajes** y se llama a `chat.completions` con temperatura controlada.

Aquí los **hechos** (pedidos, políticas) provienen de archivos JSON en `data/`, simulando lo que en producción vendría de la **base de datos** de EcoMarket (ver Fase 1).

## Configuración

1. Crea un entorno virtual e instala dependencias desde la raíz del proyecto `unidad_2`:
   `pip install -r requirements.txt`
2. Copia `.env.example` a `.env` y coloca tu clave, o define `OPENAI_API_KEY` en el sistema.
3. Ejecuta el notebook con el kernel de ese entorno.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
print("Raíz del proyecto:", ROOT)

In [ ]:
from ecomarket.client import get_chat_completion
from ecomarket.prompts import build_devolucion_messages, build_pedido_messages

## (a) Solicitud de estado de pedido

`build_pedido_messages` construye una **cadena de prompts** (datos verificados, reglas —con bloque explícito de hechos si hay retraso—, rúbrica y consulta del cliente).

- Por **número de pedido:** `order_id="ORD-00001"` (sin retraso) o `ORD-00005` (con retraso).
- Por **categoría:** omite `order_id` y usa `categoria="ropa"` (o `electronicos`, `insumos`, `equipo deporte`, etc.; se normalizan mayúsculas y acentos). Devuelve todos los pedidos de esa categoría en el JSON.

In [ ]:
user_basic = "Dame el estado de mi pedido y cuándo lo recibiré."
messages_basic = build_pedido_messages(
    order_id="ORD-00001",
    user_message=user_basic,
    pedidos_path=ROOT / "data" / "pedidos_ejemplo.json",
)
respuesta_basic = get_chat_completion(messages_basic)
print(respuesta_basic)

In [ ]:
user_improved = """Necesito el seguimiento, la fecha de entrega y el enlace para rastrear el paquete.
Si hay algún problema, explícamelo con claridad."""

messages_improved = build_pedido_messages(
    order_id="ORD-00005",
    user_message=user_improved,
    pedidos_path=ROOT / "data" / "pedidos_ejemplo.json",
)
respuesta_improved = get_chat_completion(messages_improved)
print(respuesta_improved)

In [ ]:
user_cat = "¿Qué pedidos tengo en esta categoría y cuáles van con retraso? Explica el motivo solo con los datos."
messages_cat = build_pedido_messages(
    categoria="ropa",
    user_message=user_cat,
    pedidos_path=ROOT / "data" / "pedidos_ejemplo.json",
)
respuesta_cat = get_chat_completion(messages_cat)
print(respuesta_cat)

## (b) Devolución de producto

El prompt guía al modelo usando la política en `data/politica_devoluciones.json`, diferenciando categorías que **no** suelen admitir devolución (perecederos, higiene, etc.).

In [ ]:
messages_higiene = build_devolucion_messages(
    producto="Cepillo dental eléctrico",
    categoria="Higiene personal",
    motivo="No me gustó el color después de abrir el embalaje.",
    politica_path=ROOT / "data" / "politica_devoluciones.json",
)
print(get_chat_completion(messages_higiene))

In [ ]:
messages_perecedero = build_devolucion_messages(
    producto="Bandeja de fresas orgánicas",
    categoria="Productos perecederos",
    motivo="Llegaron pasado un día de la fecha que esperaba.",
    politica_path=ROOT / "data" / "politica_devoluciones.json",
)
print(get_chat_completion(messages_perecedero))

In [ ]:
messages_ok = build_devolucion_messages(
    producto="Auriculares Bluetooth (caja sin abrir)",
    categoria="Electrónica",
    motivo="Compré dos unidades por error.",
    politica_path=ROOT / "data" / "politica_devoluciones.json",
)
print(get_chat_completion(messages_ok))